# Задание

Разработать систему хранения информации для RAG

Есть PDF документ, необходимо по нему сформировать индекс в какой-нибудь векторной бд для
последующего использования в RAG'е. Предполагается, что подобных документов может быть больше чем один. Содержание документов может меняться - соответсвенно в индексе должна лежать актуальная информация относительно текущего состояния документов.

Дополнительно:
Проработать варианты дообучения моделей поиска/ранкинга на основе структуры предоставленного документа.


## Шаг 1. Парсим PDF из примера

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# %pip install pymupdf
# %pip install qdrant-client
# %pip install sentence-transformers
# %pip install rank-bm25
# %pip install pandas numpy tqdm
# %pip install transformers

In [3]:
import pymupdf
import os
import glob
import re
import json
from tqdm.notebook import tqdm

In [4]:
DATA_DIR = './data/'

In [5]:
pdf_path = glob.glob(f'{DATA_DIR}*')[0]

In [6]:
# Открываем PDF и извлекаем текст постранично с метаинформацией
doc = pymupdf.open(pdf_path, filetype="pdf")

In [7]:
len(doc)

245

In [8]:
text_ = doc[8]

In [9]:
structured_text = []

for page_num, page in tqdm(enumerate(doc, start=1), total=len(doc)):
    blocks = page.get_text("dict")["blocks"]
    for block in blocks:
        if block['type'] != 0:  # type 0 = text
            continue
        block_text = ""
        for line in block["lines"]:
            for span in line["spans"]:
                text = span["text"].strip().replace('-\n', '')
                if not text:
                    continue

                font_size = span["size"]
                is_bold = "Bold" in span["font"]

                # Примитивная эвристика для заголовков
                if font_size > 12 or is_bold:
                    if text.lower().startswith("abstract"):
                        block_text += "\n## Abstract\n"
                    elif text.lower().startswith("introduction"):
                        block_text += "\n## Introduction\n"
                    elif text.lower().startswith("conclusion") or text.lower().startswith("discussion"):
                        block_text += f"\n## {text.strip()}\n"
                    elif text.lower().startswith("references"):
                        block_text += f"\n## References\n"
                    elif font_size > 13:
                        block_text += f"\n### {text.strip()}\n"
                    else:
                        block_text += f"\n{text.strip()}\n"
                elif text.lower().startswith("figure") or text.lower().startswith("fig."):
                    block_text += f"\n[FIGURE] {text.strip()}\n"
                elif text.lower().startswith("table"):
                    block_text += f"\n[TABLE] {text.strip()}\n"
                else:
                    block_text += f"{text} "

        if block_text.strip():
            structured_text.append(block_text.strip())

  0%|          | 0/245 [00:00<?, ?it/s]

MuPDF error: unsupported error: cannot create appearance stream for Screen annotations

MuPDF error: unsupported error: cannot create appearance stream for Screen annotations

MuPDF error: unsupported error: cannot create appearance stream for Screen annotations

MuPDF error: unsupported error: cannot create appearance stream for Screen annotations

MuPDF error: unsupported error: cannot create appearance stream for Screen annotations

MuPDF error: unsupported error: cannot create appearance stream for Screen annotations

MuPDF error: unsupported error: cannot create appearance stream for Screen annotations

MuPDF error: unsupported error: cannot create appearance stream for Screen annotations

MuPDF error: unsupported error: cannot create appearance stream for Screen annotations

MuPDF error: unsupported error: cannot create appearance stream for Screen annotations

MuPDF error: unsupported error: cannot create appearance stream for Screen annotations

MuPDF error: unsupported error: 

In [10]:
full_text = "\n".join(structured_text)
full_text[:3000]  # Покажем первые 3000 символов результата для анализа структуры

'### Конспект по обучению с подкреплением\nqbrick@mail.ru\n25 января 2022 г.\nАннотация\nСовременные алгоритмы глубокого обучения с подкреплением способны решать задачи искусственного ин- теллекта методом проб и ошибок без использования каких-либо априорных знаний о решаемой задаче. В этом конспекте собраны принципы работы основных алгоритмов, достигших прорывных результатов во многих за- дачах от игрового искусственного интеллекта до робототехники. Вся необходимая теория приводится с доказа- тельствами, использующими единый ход рассуждений, унифицированные обозначения и определения. Основная задача этой работы — не только собрать информацию из разных источников в одном месте, но понять разницу между алгоритмами различного вида и объяснить, почему они выглядят именно так, а не иначе.\nПредполагается знакомство читателя с основами машинного обучения и глубокого обучения. Об ошибках и опечатках в тексте можно сообщать в репозитории проекта .\n### arXiv:2201.09746v1  [cs.LG]  19 Jan 2022\

In [11]:
def split_into_chunks(text, max_chunk_size=512):
    """
    Разбивает текст на чанки по предложениям, не превышая max_chunk_size.
    """
    sentences = re.split(r'(?<=[.!?]) +', text)
    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= max_chunk_size:
            current_chunk += sentence + " "
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + " "

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

In [12]:
# Список чанков с мета-информацией
chunks_with_meta = []
current_section = None

for section in structured_text:
    if section.startswith("##"):
        current_section = section.replace("#", "").strip()
        continue
    elif section.startswith("###"):
        current_section = section.replace("#", "").strip()
        continue
    elif section.startswith("[FIGURE]") or section.startswith("[TABLE]"):
        chunks_with_meta.append({
            "section": current_section,
            "type": "caption",
            "content": section
        })
    else:
        # Это обычный текст — делим его на чанки
        content_chunks = split_into_chunks(section)
        for chunk in content_chunks:
            chunks_with_meta.append({
                "section": current_section,
                "type": "text",
                "content": chunk
            })

In [13]:
# Сохраняем результат
OUTPUT_DIR = f"{DATA_DIR}output/"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, os.path.split(pdf_path)[-1])
with open(output_path, "w", encoding="utf-8") as f:
    for item in chunks_with_meta:
        json.dump(item, f, ensure_ascii=False)
        f.write("\n")

In [14]:
full_text = "\n".join(structured_text)
full_text[:1000]  # Покажем первые 3000 символов результата для анализа структуры

'### Конспект по обучению с подкреплением\nqbrick@mail.ru\n25 января 2022 г.\nАннотация\nСовременные алгоритмы глубокого обучения с подкреплением способны решать задачи искусственного ин- теллекта методом проб и ошибок без использования каких-либо априорных знаний о решаемой задаче. В этом конспекте собраны принципы работы основных алгоритмов, достигших прорывных результатов во многих за- дачах от игрового искусственного интеллекта до робототехники. Вся необходимая теория приводится с доказа- тельствами, использующими единый ход рассуждений, унифицированные обозначения и определения. Основная задача этой работы — не только собрать информацию из разных источников в одном месте, но понять разницу между алгоритмами различного вида и объяснить, почему они выглядят именно так, а не иначе.\nПредполагается знакомство читателя с основами машинного обучения и глубокого обучения. Об ошибках и опечатках в тексте можно сообщать в репозитории проекта .\n### arXiv:2201.09746v1  [cs.LG]  19 Jan 2022\

In [15]:
section_filter = {chunks_with_meta[0]['section'], 'Аннотация', 'Оглавление', 'Литература'}

In [16]:
regex_content_pattern = r"^\d+(\.\d+)*\.\s+.+$"

In [17]:
re.match(pattern=regex_content_pattern, string="8.6.5 Multi-Agent DDPG (MADDPG) в смешанных играх")

In [18]:
chunks_filtered = [chunk for chunk in chunks_with_meta if (chunk['section'] not in section_filter and\
                                                           re.match(pattern=regex_content_pattern, 
                                                                    string=chunk['content']) is None)]

In [19]:
len(chunks_filtered)

5641

## Шаг 2. Индекс

In [20]:
# %pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

In [21]:
# !nvidia-smi

In [22]:
import qdrant_client
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from qdrant_client.models import Distance, VectorParams, PointStruct
import numpy as np
from typing import List, Dict

In [23]:
# %pip install accelerate

In [24]:
# 1. Настроим модель эмбеддингов и Qdrant
EMBEDDING_MODEL = "deepvk/USER-bge-m3"
VECTOR_DIM = 1024

# qdrant = qdrant_client.QdrantClient("localhost", port=6333)
qdrant = qdrant_client.QdrantClient(":memory:")
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
model = SentenceTransformer(EMBEDDING_MODEL)

In [29]:
# 2. Класс для индексации текста
class HybridSearch:
    def __init__(self, collection_name="rag_index"):
        self.collection = collection_name
        self.text_chunks = []  # Оригинальные тексты
        self.bm25 = None

        # Создаём коллекцию в Qdrant
        qdrant.recreate_collection(
            collection_name=self.collection,
            vectors_config=VectorParams(size=VECTOR_DIM, distance=Distance.COSINE),
        )

    def index_documents(self, documents: List[Dict]):
        self.text_chunks += documents
        self.bm25 = BM25Okapi([doc['content'].split() for doc in documents])

        vectors = model.encode([doc['content'] for doc in documents],
                               normalize_embeddings=True).tolist()
        points = [
            PointStruct(id=i, vector=vectors[i], payload={"text": documents[i]})
            for i in range(len(documents))
        ]
        qdrant.upsert(self.collection, points=points)

    def hybrid_search(self, query: str, lambda_weight=0.5, top_k=5):
        # BM25 поиск
        tokenized_query = query.split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        bm25_ranking = np.argsort(bm25_scores)[::-1][:top_k]

        # Векторный поиск
        query_vector = model.encode([query])[0].tolist()
        search_result = qdrant.search(
            collection_name=self.collection,
            query_vector=query_vector,
            limit=top_k
        )

        # Объединение результатов
        results = {}
        for i, idx in enumerate(bm25_ranking):
            results[idx] = lambda_weight * bm25_scores[idx]
        for i, hit in enumerate(search_result):
            idx = int(hit.id)
            results[idx] = results.get(idx, 0) + (1 - lambda_weight) * hit.score

        # Финальное ранжирование
        ranked_results = sorted(results.items(), key=lambda x: x[1], reverse=True)
        return [{'item': self.text_chunks[idx], 'score': _} for idx, _ in ranked_results]
    
    
    def update_document(self, doc_id: int, new_text: str):
        if 0 <= doc_id < len(self.text_chunks):
            self.text_chunks[doc_id] = new_text
            tokenized_doc = new_text.split()
            self.bm25 = BM25Okapi(tokenized_doc)
            vector = model.encode([new_text], normalize_embeddings=True).tolist()[0]
            point = PointStruct(id=doc_id, vector=vector, payload={"text": new_text})
            qdrant.upsert(self.collection, points=[point])
        else:
            raise IndexError("Document ID is out of range")

    def delete_document(self, doc_id: int):
        if 0 <= doc_id < len(self.text_chunks):
            qdrant.delete(self.collection, points_selector={"points": [doc_id]})
            self.text_chunks[doc_id] = ""
            tokenized_docs = [doc.split() for doc in self.text_chunks if doc.strip() != ""]
            self.bm25 = BM25Okapi(tokenized_docs)
        else:
            raise IndexError("Document ID is out of range")

In [30]:
# Пример использования
hybrid_search = HybridSearch()
documents = chunks_filtered

C:\Users\administrator\AppData\Local\Temp\ipykernel_12680\3875965356.py:9: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


In [31]:
%%time
hybrid_search.index_documents(documents)

CPU times: total: 1h 27min 37s
Wall time: 45min 12s


In [32]:
%%time
query = "бенчмарк atari"
results = hybrid_search.hybrid_search(query)
print(results)

[{'item': {'section': 'Задача обучения с подкреплением', 'type': 'text', 'content': 'Пример 23 — Иг ры Atari : Atari — набор из 57 игр с дискретным пространством действий. Наблюдением является экран видео-игры (изображение), у агента имеется до 18 действий (в некоторых играх действия, соответству- ющие бездействующим кнопкам джойстика, по умолчанию убраны). Награда — счёт в игре. Визуализация игр из OpenAI Gym .'}, 'score': 0.28460385908999597}, {'item': {'section': 'Задача обучения с подкреплением', 'type': 'text', 'content': 'Пример 24 — Atar i RAM : Игры Atari представлены в ещё одной версии — «RAM-версии». Состоянием считается не изображение экрана, а 128 байт памяти Atari-консоли, в которой содержится вся информация, необходимая для расчёта игры (координаты игрока и иные параметры). По определению, такое состояние «полностью наблюдаемое», и также может использоваться для тестирования алгоритмов.'}, 'score': 0.27163322105781706}, {'item': {'section': 'Policy Gradient подход', 'type

C:\Users\administrator\AppData\Local\Temp\ipykernel_12680\3875965356.py:34: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_result = qdrant.search(


## Пробуем то же самое, но импортируя из модуля

In [69]:
from utils.pdfparser import PDFParser
from utils.chunker import TextChunker

In [ ]:
pdf_path = f"{DATA_DIR}2201.09746v1.pdf"
OUTPUT_DIR = f"{DATA_DIR}output/"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, os.path.splitext(os.path.split(pdf_path)[-1])[0]+'.json')

parser = PDFParser()
parser.load_text(pdf_path)
parser.parse()

In [70]:
chunker = TextChunker(parser.get_structured_blocks())
chunker.chunk()
chunker.filter_chunks()
chunker.save(output_path)

In [72]:
# Пример использования
hybrid_search = HybridSearch()
documents = chunker.chunks

C:\Users\administrator\AppData\Local\Temp\ipykernel_12680\3875965356.py:9: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


In [ ]:
%%time
hybrid_search.index_documents(documents)

In [ ]:
%%time
query = "бенчмарк atari"
results = hybrid_search.hybrid_search(query)
print(results)